# LintBench — HPC Inference (Open-Source Models)

Drives LintBench inference against locally-hosted vLLM servers.

| Short name | HuggingFace ID | Default port | GPUs (A100-80 GB) | Reasoning |
|---|---|---|---|---|
| `qwen3-coder-30b` | `Qwen/Qwen3-Coder-30B-A3B` | 8000 | 2 | Yes |
| `llama3-70b-instruct` | `meta-llama/Meta-Llama-3-70B-Instruct` | 8001 | 4 | No |
| `deepseek-r1-32b` | `deepseek-ai/DeepSeek-R1-Distill-Qwen-32B` | 8002 | 2 | Yes |
| `gemma4-26b` | `google/gemma-4-26b-it` | 8003 | 1 | Yes |

**Prerequisites**
1. Launch a vLLM server via the corresponding SLURM script in this directory.
2. Note the hostname and port (written to `logs/*_endpoint.txt` next to the SLURM script).
3. Run cells from the repo root, or let cell 1 set the working directory automatically.

---

In [ ]:
import os, sys
from pathlib import Path

# This notebook lives at lint_benchmark/inference/hpc/; repo root is three levels up.
REPO_ROOT      = Path("../../..").resolve()
LINTBENCH_ROOT = REPO_ROOT / "lint_benchmark"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# CWD must be lint_benchmark/ so relative paths (data/, generated/) resolve correctly.
os.chdir(LINTBENCH_ROOT)
print("CWD:", os.getcwd())

In [ ]:
from lint_benchmark.inference.hpc import HPC_MODELS, HPCInferenceConfig, run_hpc_inference, wait_for_server

print("Registered HPC models:")
for name, meta in HPC_MODELS.items():
    tag = "reasoning" if meta["reasoning"] else "instruct"
    print(f"  {name:30s}  port={meta['default_port']}  gpus={meta['gpus']}  [{tag}]")

## 1  Configuration

In [ ]:
# ── Edit these ────────────────────────────────────────────────────────────────
VLLM_HOST  = "localhost"   # compute node hostname where vLLM is running
BENCHMARK  = "data/dataset.jsonl"
OUT_DIR    = "generated"
LIMIT      = None          # set to e.g. 10 for a quick smoke-test
MAX_TOKENS = 32768
# ─────────────────────────────────────────────────────────────────────────────

PROMPTS = [
    "zero_shot",
    "api_hint",
    "skeleton",
    "few_shot_surface_matched",
]

## 2  Qwen3-Coder-30B  *(reasoning)*

Launch: `sbatch inference/hpc/slurm_vllm_qwen3_coder_30b.sh`

In [ ]:
MODEL = "qwen3-coder-30b"
PORT  = HPC_MODELS[MODEL]["default_port"]

if wait_for_server(VLLM_HOST, PORT, timeout=60):
    results_qwen = {}
    for prompt in PROMPTS:
        cfg = HPCInferenceConfig(
            model=MODEL, host=VLLM_HOST, port=PORT,
            prompt=prompt, out=OUT_DIR, benchmark=BENCHMARK,
            max_tokens=MAX_TOKENS, limit=LIMIT,
            capture_reasoning=HPC_MODELS[MODEL]["reasoning"],
        )
        results_qwen[prompt] = run_hpc_inference(cfg)
        print(f"[{MODEL}/{prompt}] → {results_qwen[prompt]}")
else:
    print("Server not ready — skipping Qwen3-Coder-30B")

## 3  LLaMA 3-70B-Instruct  *(instruct)*

Launch: `sbatch inference/hpc/slurm_vllm_llama3_70b.sh`  
Requires `HF_TOKEN` (gated model).

In [ ]:
MODEL = "llama3-70b-instruct"
PORT  = HPC_MODELS[MODEL]["default_port"]

if wait_for_server(VLLM_HOST, PORT, timeout=60):
    results_llama = {}
    for prompt in PROMPTS:
        cfg = HPCInferenceConfig(
            model=MODEL, host=VLLM_HOST, port=PORT,
            prompt=prompt, out=OUT_DIR, benchmark=BENCHMARK,
            max_tokens=MAX_TOKENS, limit=LIMIT,
            capture_reasoning=HPC_MODELS[MODEL]["reasoning"],
        )
        results_llama[prompt] = run_hpc_inference(cfg)
        print(f"[{MODEL}/{prompt}] → {results_llama[prompt]}")
else:
    print("Server not ready — skipping LLaMA 3-70B-Instruct")

## 4  DeepSeek-R1-Distill-Qwen-32B  *(reasoning)*

Launch: `sbatch inference/hpc/slurm_vllm_deepseek_r1_32b.sh`

In [ ]:
MODEL = "deepseek-r1-32b"
PORT  = HPC_MODELS[MODEL]["default_port"]

if wait_for_server(VLLM_HOST, PORT, timeout=60):
    results_ds = {}
    for prompt in PROMPTS:
        cfg = HPCInferenceConfig(
            model=MODEL, host=VLLM_HOST, port=PORT,
            prompt=prompt, out=OUT_DIR, benchmark=BENCHMARK,
            max_tokens=MAX_TOKENS, limit=LIMIT,
            capture_reasoning=HPC_MODELS[MODEL]["reasoning"],
        )
        results_ds[prompt] = run_hpc_inference(cfg)
        print(f"[{MODEL}/{prompt}] → {results_ds[prompt]}")
else:
    print("Server not ready — skipping DeepSeek-R1-Distill-Qwen-32B")

## 5  Gemma 4-26B-A4B-IT  *(reasoning)*

Launch: `sbatch inference/hpc/slurm_vllm_gemma4_26b.sh`  
Requires `HF_TOKEN` (gated model). MoE — fits on 1×A100-80GB.

In [ ]:
MODEL = "gemma4-26b"
PORT  = HPC_MODELS[MODEL]["default_port"]

if wait_for_server(VLLM_HOST, PORT, timeout=60):
    results_gemma = {}
    for prompt in PROMPTS:
        cfg = HPCInferenceConfig(
            model=MODEL, host=VLLM_HOST, port=PORT,
            prompt=prompt, out=OUT_DIR, benchmark=BENCHMARK,
            max_tokens=MAX_TOKENS, limit=LIMIT,
            capture_reasoning=HPC_MODELS[MODEL]["reasoning"],
        )
        results_gemma[prompt] = run_hpc_inference(cfg)
        print(f"[{MODEL}/{prompt}] → {results_gemma[prompt]}")
else:
    print("Server not ready — skipping Gemma 4-26B")

## 6  Run Evaluation

In [ ]:
import subprocess, shlex

RUN_ID = None   # None → eval all runs in OUT_DIR

cmd = ["bash", "run_eval_all.sh", "--generated", OUT_DIR, "--results", "results"]
if RUN_ID:
    cmd += ["--run-id", RUN_ID]

print("Running:", shlex.join(cmd))
subprocess.run(cmd, check=True)

## 7  Summarise Results

In [ ]:
import json, glob
import pandas as pd

rows = []
for path in sorted(glob.glob("results/**/*.json", recursive=True)):
    try:
        data = json.loads(Path(path).read_text())
    except Exception:
        continue
    meta    = data.get("metadata", {})
    metrics = data.get("metrics",  {})
    rows.append({
        "model":          meta.get("model", ""),
        "prompt":         meta.get("prompt", ""),
        "n_instances":    metrics.get("n_instances", 0),
        "pass@1":         metrics.get("pass_at_1", metrics.get("pass@1", None)),
        "compile_rate":   metrics.get("compile_rate",   None),
        "execution_rate": metrics.get("execution_rate", None),
    })

df = pd.DataFrame(rows)
df_hpc = df[df["model"].isin(HPC_MODELS)]
df_hpc.sort_values(["model", "prompt"]).style.format(
    {"pass@1": "{:.1%}", "compile_rate": "{:.1%}", "execution_rate": "{:.1%}"}
)

## 8  Inspect Generation Logs

In [ ]:
import glob, json
import pandas as pd

log_records = []
for log_path in glob.glob(f"{OUT_DIR}/**/generation_log.jsonl", recursive=True):
    for line in open(log_path):
        rec = json.loads(line)
        if rec.get("model") in HPC_MODELS:
            log_records.append(rec)

df_logs = pd.DataFrame(log_records)
print(f"Total log records: {len(df_logs)}")

if not df_logs.empty and "usage" in df_logs.columns:
    df_logs["in_tok"]  = df_logs["usage"].apply(lambda u: (u or {}).get("input_tokens",  0))
    df_logs["out_tok"] = df_logs["usage"].apply(lambda u: (u or {}).get("output_tokens", 0))
    has_reasoning = df_logs["reasoning_content"].notna() if "reasoning_content" in df_logs.columns else False
    summary = (
        df_logs.groupby(["model", "prompt"])
        .agg(calls=("success", "count"),
             successes=("success", "sum"),
             in_tokens=("in_tok", "sum"),
             out_tokens=("out_tok", "sum"))
        .reset_index()
    )
    display(summary)

## 9  Smoke-test (stub mode — no GPU required)

In [ ]:
cfg_stub = HPCInferenceConfig(
    model="qwen3-coder-30b",
    prompt="zero_shot",
    out="generated_smoke",
    benchmark=BENCHMARK,
    limit=5,
    stub=True,
)
result = run_hpc_inference(cfg_stub)
print(result)